Count class IDs per dataset

In [5]:
import os
from collections import Counter

base = r"E:\sem7\FYP\9_24\objectdetetction\datasets"
folders = ["door", "fire_extinguisher_yolov11", "gauges_yolov11"]  
for f in folders:
    total_counts = Counter()
    for subset in ["train", "valid", "test"]:
        label_dir = os.path.join(base, f, subset, "labels")
        if not os.path.exists(label_dir):
            continue
        counts = Counter()
        for file in os.listdir(label_dir):
            if not file.endswith(".txt"):
                continue
            with open(os.path.join(label_dir, file)) as lab:
                for line in lab:
                    if line.strip():
                        cls_id = line.split()[0]
                        counts[cls_id] += 1
                        total_counts[cls_id] += 1
        print(f"{f}/{subset} → class IDs: {dict(counts)}")
    print(f"Total for {f}: {dict(total_counts)}\n")


door/train → class IDs: {'0': 1361}
door/valid → class IDs: {'0': 328}
door/test → class IDs: {'0': 238}
Total for door: {'0': 1927}

fire_extinguisher_yolov11/train → class IDs: {'0': 5615}
fire_extinguisher_yolov11/valid → class IDs: {'0': 606}
Total for fire_extinguisher_yolov11: {'0': 6221}

gauges_yolov11/train → class IDs: {'0': 3212}
gauges_yolov11/valid → class IDs: {'0': 687}
gauges_yolov11/test → class IDs: {'0': 350}
Total for gauges_yolov11: {'0': 4249}



Remap class IDs to 0, 1, 2 only

In [8]:
import os

base = r"E:\sem7\FYP\9_24\objectdetetction\datasets"

# New mapping (no switch_panel)
dataset_to_newid = {
    "fire_extinguisher_yolov11": 0,
    "door": 1,
    "gauges_yolov11": 2,
}

for dataset, new_id in dataset_to_newid.items():
    print(f"\nFixing {dataset} → class ID {new_id}")
    for subset in ["train", "valid", "test"]:
        label_dir = os.path.join(base, dataset, subset, "labels")
        if not os.path.exists(label_dir):
            continue
        for file in os.listdir(label_dir):
            if not file.endswith(".txt"):
                continue
            path = os.path.join(label_dir, file)
            with open(path, "r") as f:
                lines = f.readlines()
            new_lines = []
            for line in lines:
                parts = line.strip().split()
                if not parts:
                    continue
                parts[0] = str(new_id)
                new_lines.append(" ".join(parts))
            with open(path, "w") as f:
                f.write("\n".join(new_lines))
    print(f" {dataset} fixed.")



Fixing fire_extinguisher_yolov11 → class ID 0
 fire_extinguisher_yolov11 fixed.

Fixing door → class ID 1
 door fixed.

Fixing gauges_yolov11 → class ID 2
 gauges_yolov11 fixed.


Count again

In [9]:
import os
from collections import Counter

base = r"E:\sem7\FYP\9_24\objectdetetction\datasets"
folders = ["door", "fire_extinguisher_yolov11", "gauges_yolov11"]

for f in folders:
    total = Counter()
    for subset in ["train", "valid", "test"]:
        label_dir = os.path.join(base, f, subset, "labels")
        if not os.path.exists(label_dir):
            continue
        for file in os.listdir(label_dir):
            if not file.endswith(".txt"):
                continue
            with open(os.path.join(label_dir, file)) as lab:
                for line in lab:
                    if line.strip():
                        total[line.split()[0]] += 1
    print(f"{f} → {dict(total)}")


door → {'1': 1927}
fire_extinguisher_yolov11 → {'0': 6221}
gauges_yolov11 → {'2': 4249}


Merge into one

In [11]:
from sklearn.model_selection import train_test_split
import glob, shutil, os

base = r"E:\sem7\FYP\9_24\objectdetetction\datasets"
merged = os.path.join(base, "custom_merged_obb")

# Clean + create directories
for split in ["train", "valid"]:
    os.makedirs(os.path.join(merged, f"images/{split}"), exist_ok=True)
    os.makedirs(os.path.join(merged, f"labels/{split}"), exist_ok=True)

# Collect all label files from the 3 datasets
all_labels = []
for d in ["door", "fire_extinguisher_yolov11", "gauges_yolov11"]:
    for split in ["train", "valid"]:
        all_labels += glob.glob(os.path.join(base, d, split, "labels", "*.txt"))

# Split for final train/valid
train, val = train_test_split(all_labels, test_size=0.2, random_state=42)

# Copy labels + images
for subset, files in [("train", train), ("valid", val)]:
    for label in files:
        img = label.replace("labels", "images").replace(".txt", ".jpg")
        if not os.path.exists(img):
            img = img.replace(".jpg", ".png")
        shutil.copy(label, os.path.join(merged, f"labels/{subset}"))
        shutil.copy(img, os.path.join(merged, f"images/{subset}"))

print("Merged dataset ready:", merged)


Merged dataset ready: E:\sem7\FYP\9_24\objectdetetction\datasets\custom_merged_obb


Final count check of merged dataset

In [12]:
import os
from collections import Counter

merged = r"E:\sem7\FYP\9_24\objectdetetction\datasets\custom_merged_obb"
for split in ["train", "valid"]:
    label_dir = os.path.join(merged, f"labels/{split}")
    total = Counter()
    for file in os.listdir(label_dir):
        with open(os.path.join(label_dir, file)) as f:
            for line in f:
                if line.strip():
                    total[line.split()[0]] += 1
    print(f"{split} → {dict(total)}")


train → {'0': 4980, '2': 3117, '1': 1331}
valid → {'0': 1241, '2': 782, '1': 358}


bettter method to merge

In [2]:
import os, glob, shutil
from collections import Counter

base = r"E:\sem7\FYP\9_24\objectdetetction\datasets"
merged = os.path.join(base, "custom_merged_obb")

datasets = {
    "door": "door",
    "fire_extinguisher_yolov11": "fire",
    "gauges_yolov11": "gauge"
}

# Create output structure
for split in ["train", "valid"]:
    os.makedirs(os.path.join(merged, f"images/{split}"), exist_ok=True)
    os.makedirs(os.path.join(merged, f"labels/{split}"), exist_ok=True)

def copy_pair(img_path, label_path, split, prefix):
    """Copy image+label safely with renamed prefix"""
    ext = os.path.splitext(img_path)[1]
    new_name = f"{prefix}_{os.path.basename(label_path)}"
    new_img_name = new_name.replace(".txt", ext)

    shutil.copy(label_path, os.path.join(merged, f"labels/{split}", new_name))
    shutil.copy(img_path, os.path.join(merged, f"images/{split}", new_img_name))

# Merge process
for split in ["train", "valid"]:
    print(f"\n🔹 Merging {split} data...")
    for folder, prefix in datasets.items():
        label_dir = os.path.join(base, folder, split, "labels")
        img_dir = os.path.join(base, folder, split, "images")
        if not os.path.exists(label_dir):
            continue

        count = 0
        for label_path in glob.glob(os.path.join(label_dir, "*.txt")):
            with open(label_path) as f:
                lines = [line.strip() for line in f if line.strip()]
            if not lines:
                continue  # skip empty labels

            # Find matching image
            img_path = os.path.join(img_dir, os.path.basename(label_path).replace(".txt", ".jpg"))
            if not os.path.exists(img_path):
                img_path = img_path.replace(".jpg", ".png")
            if not os.path.exists(img_path):
                continue

            copy_pair(img_path, label_path, split, prefix)
            count += 1
        print(f"  {folder}: copied {count} labeled images.")

# Summary check
print("\n Merge complete! Summary:")
for split in ["train", "valid"]:
    lbl_dir = os.path.join(merged, f"labels/{split}")
    total = Counter()
    for file in os.listdir(lbl_dir):
        with open(os.path.join(lbl_dir, file)) as f:
            for line in f:
                if line.strip():
                    total[line.split()[0]] += 1
    print(f"  {split}: {sum(total.values())} boxes, class counts = {dict(total)}")

print("\nMerged dataset ready at:", merged)



🔹 Merging train data...
  door: copied 1350 labeled images.
  fire_extinguisher_yolov11: copied 2934 labeled images.
  gauges_yolov11: copied 2763 labeled images.

🔹 Merging valid data...
  door: copied 322 labeled images.
  fire_extinguisher_yolov11: copied 328 labeled images.
  gauges_yolov11: copied 653 labeled images.

 Merge complete! Summary:
  train: 10188 boxes, class counts = {'0': 10188}
  valid: 1621 boxes, class counts = {'0': 1621}

Merged dataset ready at: E:\sem7\FYP\9_24\objectdetetction\datasets\custom_merged_obb


In [4]:
import glob, os
from collections import Counter

base = "E:\sem7\FYP\9_24\objectdetetction\datasets\custom_merged_obb"
for split in ["train", "valid"]:
    counter = Counter()
    for f in glob.glob(os.path.join(base, f"labels", split, "*.txt")):
        with open(f) as file:
            for line in file:
                if line.strip():
                    counter[line.strip().split()[0]] += 1
    print(f"{split} → {counter}")


train → Counter({'0': 10188})
valid → Counter({'0': 1621})
